# Exploratory Data Analysis: Bug Report Severity Dataset

This notebook looks at the Bugzilla / Eclipse bug report dataset used in the Bug Report AI project.
It checks the size and columns, missing values, class balance, text length, exact duplicates, conflicting labels and the most common words.
Every chart is also saved as a PNG file in the `eda_figures` folder, so you can use it in the course report.

**How to use:** set `DATA_PATH` in the first code cell, then run all cells (Kernel / Run All).

## 1. Setup and data path

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Settings: change these ----------------------------------------------
DATA_PATH = r"ml/data/raw/bugzilla_eclipse/filtered_data.xlsx - Sheet1.csv"   # .csv, .parquet, .json or .jsonl
TEXT_COL = None       # name of the bug report text column, or None to detect it
SEVERITY_COL = None   # name of the severity label column, or None to detect it
PROJECT_COL = None    # name of the project/product column, or None to detect it (optional)
# ---------------------------------------------------------------------------

KNOWN_LABELS = ["blocker", "critical", "major", "normal", "minor", "trivial"]
FIG_DIR = Path("eda_figures")
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
})

def save_fig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

## 2. Load the data

In [ ]:
path = Path(DATA_PATH)
if not path.exists():
    raise FileNotFoundError(
        f"File not found: {path}. Set DATA_PATH in the setup cell to your data file."
    )

suffix = path.suffix.lower()
if suffix == ".csv":
    df = pd.read_csv(path)
elif suffix == ".parquet":
    df = pd.read_parquet(path)
elif suffix == ".json":
    df = pd.read_json(path)
elif suffix == ".jsonl":
    df = pd.read_json(path, lines=True)
else:
    raise ValueError(f"Unsupported file type: {suffix}")

print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
display(df.dtypes.to_frame("dtype"))
display(df.head())

## 3. Find the important columns

In [ ]:
def is_text_col(series):
    return pd.api.types.is_string_dtype(series) or pd.api.types.is_object_dtype(series)

text_cols = [c for c in df.columns if is_text_col(df[c])]

if SEVERITY_COL is None:
    best, best_hits = None, 0
    for c in text_cols:
        values = df[c].dropna().astype(str).str.strip().str.lower().unique()
        if len(values) > 20:
            continue
        hits = sum(v in KNOWN_LABELS for v in values)
        if hits > best_hits:
            best, best_hits = c, hits
    SEVERITY_COL = best

if TEXT_COL is None:
    candidates = [c for c in text_cols if c != SEVERITY_COL]
    TEXT_COL = max(candidates, key=lambda c: df[c].dropna().astype(str).str.len().mean())

if PROJECT_COL is None:
    for c in df.columns:
        if c not in (TEXT_COL, SEVERITY_COL) and re.search("project|product|component", str(c), re.I):
            PROJECT_COL = c
            break

if SEVERITY_COL is None:
    raise ValueError("Could not detect the severity column. Set SEVERITY_COL in the setup cell.")

print("Text column     :", TEXT_COL)
print("Severity column :", SEVERITY_COL)
print("Project column  :", PROJECT_COL)

## 4. Missing and empty values

In [ ]:
missing = df.isna().sum()
empty = pd.Series(
    {c: int((df[c].astype(str).str.strip() == "").sum()) if is_text_col(df[c]) else 0 for c in df.columns}
)
quality = pd.DataFrame({"missing": missing, "empty_text": empty})
quality["percent_missing"] = (quality["missing"] / len(df) * 100).round(3)
display(quality)

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(quality.index, quality["missing"], color="#2563eb")
ax.set_xlabel("Number of missing values")
ax.set_title("Missing values per column")
for bar, value in zip(bars, quality["missing"]):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2, f" {value:,}", va="center", fontsize=9)
ax.invert_yaxis()
save_fig("01_missing_values")
plt.show()

## 5. Class balance (severity labels)

In [ ]:
labels = df[SEVERITY_COL].astype(str).str.strip().str.lower()
counts = labels.value_counts()
percent = (counts / counts.sum() * 100).round(2)
balance = pd.DataFrame({"count": counts, "percent": percent})
display(balance)

imbalance = counts.max() / counts.min()
print(f"Largest class: {counts.index[0]} ({percent.iloc[0]}%)")
print(f"Smallest class: {counts.index[-1]} ({percent.iloc[-1]}%)")
print(f"Imbalance ratio (largest / smallest): {imbalance:,.1f} to 1")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(counts.index, counts.values, color="#2563eb")
axes[0].set_title("Reports per severity")
axes[0].set_ylabel("Number of reports")
for i, (v, p) in enumerate(zip(counts.values, percent.values)):
    axes[0].text(i, v, f"{p}%", ha="center", va="bottom", fontsize=9)
axes[1].bar(counts.index, counts.values, color="#7c3aed")
axes[1].set_yscale("log")
axes[1].set_title("Same chart, log scale (small classes visible)")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
save_fig("02_class_balance")
plt.show()

## 6. Text length

In [ ]:
text = df[TEXT_COL].fillna("").astype(str)
chars = text.str.len()
words = text.str.split().str.len()

length_stats = pd.DataFrame({"characters": chars.describe(), "words": words.describe()}).round(1)
display(length_stats)

clip = chars.quantile(0.99)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(chars[chars <= clip], bins=60, color="#2563eb")
axes[0].set_title("Text length in characters (up to the 99th percentile)")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Number of reports")

order = [c for c in KNOWN_LABELS if c in counts.index] + [c for c in counts.index if c not in KNOWN_LABELS]
groups = [chars[labels == c].clip(upper=clip) for c in order]
axes[1].boxplot(groups, tick_labels=order, showfliers=False)
axes[1].set_title("Text length by severity")
axes[1].set_ylabel("Characters")
axes[1].tick_params(axis="x", rotation=30)
save_fig("03_text_length")
plt.show()

print(f"Reports with fewer than 20 characters: {(chars < 20).sum():,}")

## 7. Duplicates and conflicting labels

In [ ]:
def normalise(s):
    return s.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

norm_text = normalise(text)

exact_row_dups = int(df.duplicated().sum())
dup_text_mask = norm_text.duplicated(keep=False) & (norm_text != "")
extra_copies = int(norm_text.duplicated().sum())

print(f"Fully identical rows                  : {exact_row_dups:,}")
print(f"Rows whose text appears more than once: {int(dup_text_mask.sum()):,}")
print(f"Extra copies (beyond the first)       : {extra_copies:,}")

# Same text, different severity: these are conflicting labels
check = pd.DataFrame({"text": norm_text[dup_text_mask], "label": labels[dup_text_mask]})
per_text = check.groupby("text")["label"].nunique()
conflicts = int((per_text > 1).sum())
print(f"Texts that appear with more than one severity: {conflicts:,}")

if conflicts:
    example = per_text[per_text > 1].index[0]
    print("\nExample of a conflicting text (shortened):")
    print(example[:200])
    print("Labels:", sorted(check[check["text"] == example]["label"].unique()))

## 8. Projects (if the data has a project column)

In [ ]:
if PROJECT_COL is None:
    print("No project column found, skipping this section.")
else:
    top = df[PROJECT_COL].fillna("(missing)").astype(str).value_counts().head(10)
    display(top.to_frame("reports"))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(top.index[::-1], top.values[::-1], color="#2563eb")
    ax.set_title(f"Top 10 values of '{PROJECT_COL}'")
    ax.set_xlabel("Number of reports")
    save_fig("04_top_projects")
    plt.show()

## 9. Most common words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

sample_size = min(20000, len(df))
sample_idx = np.random.RandomState(42).choice(len(df), size=sample_size, replace=False)
sample_text = text.iloc[sample_idx]
sample_labels = labels.iloc[sample_idx].to_numpy()

vectorizer = CountVectorizer(stop_words="english", max_features=5000, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")
matrix = vectorizer.fit_transform(sample_text)
vocab = np.array(vectorizer.get_feature_names_out())

overall = np.asarray(matrix.sum(axis=0)).ravel()
top_idx = overall.argsort()[::-1][:15]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(vocab[top_idx][::-1], overall[top_idx][::-1], color="#2563eb")
ax.set_title(f"15 most common words (sample of {sample_size:,} reports)")
ax.set_xlabel("Occurrences")
save_fig("05_top_words")
plt.show()

per_class = {}
for label in order:
    mask = sample_labels == label
    if mask.sum() == 0:
        continue
    class_counts = np.asarray(matrix[mask].sum(axis=0)).ravel()
    per_class[label] = list(vocab[class_counts.argsort()[::-1][:8]])
display(pd.DataFrame.from_dict(per_class, orient="index", columns=[f"word {i+1}" for i in range(8)]))

## 10. Summary

In [ ]:
summary = {
    "Rows": f"{len(df):,}",
    "Columns": df.shape[1],
    "Missing values (total)": f"{int(quality['missing'].sum()):,}",
    "Number of severity classes": len(counts),
    "Largest class": f"{counts.index[0]} ({percent.iloc[0]}%)",
    "Smallest class": f"{counts.index[-1]} ({percent.iloc[-1]}%)",
    "Imbalance ratio": f"{imbalance:,.1f} to 1",
    "Median text length (characters)": f"{chars.median():,.0f}",
    "Extra copies of the same text": f"{extra_copies:,}",
    "Texts with conflicting severity": f"{conflicts:,}",
}
display(pd.Series(summary, name="value").to_frame())
print("Charts saved in:", FIG_DIR.resolve())

## Notes for the report

- A strongly imbalanced dataset makes accuracy misleading, so the project reports **macro F1**, which treats every class equally.
- Identical texts with different severities show that the labels are noisy, which limits how good any model can become.
- This notebook only looks at the data. It does not change or clean it.